# Debbi — BPE + AICL Fork • 85% Training on Colab T4

1-click training for **Debbi-150M** with the **85% BPE+Phrase tokenizer** (`bpe_fork` branch).
- BPE 6k `sp_bpe_6k.model` + 4k phrases = 10k vocab, 6.78 c/t, 85.2% on `bpe_corpus` (90.3% with 60k)
- Honest held-out 84.5% (10k) vs BPE 77.8% — see `benchmark_heldout.png`

**Run all cells → T4 trains → checkpoints in `checkpoints/debbi-150m/`**

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch; print(f"cuda: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"torch: {torch.__version__}")

## 1. Clone bpe-fork and install

In [ ]:
!rm -rf AICL-Debbi
!git clone https://github.com/vspcoderz/AICL-Debbi.git -b bpe-fork --depth 1
%cd AICL-Debbi
!pip install -q -r requirements.txt sentencepiece
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!ls tokenizer/vocabularies/sp_bpe_6k.model tokenizer/vocabularies/bpe_phrase_4k.json
print("ready: BPE 6k + phrase 4k (10k vocab)")

## 2. Prepare data — BPE+Phrase 85% (default)
Uses `bpe_phrase_4k.json` (85.2% pure, 84.5% held-out). For 90% use `bpe_phrase_60k.json` (66k vocab).
Expects your training text at `traningtext.txt` or uses `bpe_corpus.txt` + `traningtext.txt` combined.

In [ ]:
# Robust: find training text (Colab has no /root/aicl, use repo files or upload)
import os
candidates = ["/root/aicl/traningtext.txt", "/tmp/opencode/aiclcorpus/bpe_corpus.txt", "traningtext.txt", "data/traningtext.txt"]
INPUT = next((p for p in candidates if os.path.exists(p)), None)
if INPUT is None:
    print("No traningtext found, building fallback corpus from repo .py files...")
    import subprocess
    subprocess.run("find . -type f -name '*.py' | head -30 | xargs cat > /tmp/combined_fallback.txt", shell=True)
    # Ensure at least 50k chars for demo (repeat README if needed)
    txt = open('/tmp/combined_fallback.txt').read()
    if len(txt) < 50000:
        txt += open('README.md').read() * 200
        open('/tmp/combined_fallback.txt','w').write(txt)
    INPUT = "/tmp/combined_fallback.txt"
print(f"INPUT={INPUT} chars={len(open(INPUT).read())}")
if len(open(INPUT).read()) < 1000:
    print("WARNING: corpus too small (<1k), upload traningtext.txt via Files > Upload")

!python data/prepare_data.py --input $INPUT --out-dir data --tokenizer bpe_phrase --sp-model tokenizer/vocabularies/sp_bpe_6k.model --phrase-vocab tokenizer/vocabularies/bpe_phrase_4k.json
!ls -lh data/corpus.bin data/id_map.json
!cat data/id_map.json | grep -E "vocab_size|tokenizer|num_tokens" | head

## 3. Train Debbi-150M
Config: 768 dim / 12 layers / 12 heads / 1024 seq len / 10k vocab → ~150M. bf16 on T4, grad checkpointing on.

In [ ]:
!python model/train.py --max-steps 20000 --batch-size 8 --grad-accum 1 2>&1 | tee train.log
print("done — checkpoints in checkpoints/debbi-150m/")

## 4. Generate sample

In [ ]:
!python model/generate.py --ckpt checkpoints/debbi-150m/last.pt --id-map data/id_map.json --prompt "def quicksort(arr):" --max-new-tokens 128


---
*For 90% (60k phrases, 66k vocab, int32):* change `--phrase-vocab` to `bpe_phrase_60k.json` in the prepare step. Needs more VRAM.
*To use pure BPE 78%: `--tokenizer bpe`*
*See `benchmark_heldout.png` for honest 80/20 held-out numbers.*